In [18]:

import os
from pathlib import Path


import pandas as pd
import torch
import torch.nn.functional as F
from torch import Tensor
from transformers import AutoTokenizer, AutoModel

from datasets import load_dataset

In [21]:
def average_pool(last_hidden_states: Tensor,
                 attention_mask: Tensor) -> Tensor:
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]

In [19]:
tokenizer = AutoTokenizer.from_pretrained('intfloat/multilingual-e5-base')
model = AutoModel.from_pretrained('intfloat/multilingual-e5-base')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [25]:
dataset_docs = load_dataset('unicamp-dl/mmarco', 'collection-portuguese', streaming=True, trust_remote_code=True)
# Pega só os primeiros 100 exemplos
docs = list(dataset_docs['collection'].take(100))
# Separa o texto
collection_textos = [item['text'] for item in docs]

docs_batch = tokenizer(collection_textos, max_length=512, padding=True, truncation=True, return_tensors='pt')

Repo card metadata block was not found. Setting CardData to empty.


In [26]:
dataset_queries = load_dataset('unicamp-dl/mmarco', 'queries-portuguese', streaming=True, trust_remote_code=True)
# Pega só os primeiros 100 exemplos
amostra = list(dataset_queries['train'].take(100))
# Separa o texto
queries_textos = [item['text'] for item in amostra]

queries_batch = tokenizer(queries_textos, max_length=512, padding=True, truncation=True, return_tensors='pt')

Repo card metadata block was not found. Setting CardData to empty.


In [ ]:
outputs = model(**batch_dict)
docs_embeddings = average_pool(outputs.last_hidden_state, batch_dict['attention_mask'])
docs_embeddings = F.normalize(docs_embeddings, p=2, dim=1)

#print(outputs)

#print(docs_embeddings)

print(docs_embeddings.shape)
